# 05 — The Inner Product & Dirac Notation

Learn the one piece of algebra every later notebook leans on: the inner product $\langle\varphi|\psi\rangle$ that turns two states into a single number.

**Prerequisites:** `01/01_amplitudes_and_superposition`, `01/04_measurement_and_born_rule`.

**What you'll be able to do**
- Form a *bra* $\langle\varphi|$ as the conjugate transpose of a *ket* $|\varphi\rangle$, and compute the overlap $\langle\varphi|\psi\rangle$.
- Recognise the Born rule as a squared overlap, $P(x) = |\langle x|\psi\rangle|^2$.
- Use orthonormality and read $|\langle\varphi|\psi\rangle|$ as a measure of how alike two states are.

You already multiply amplitudes and take their squared sizes. The inner product packages those moves into one operation that the rest of the course speaks fluently — it is how we ask "how much of $|\psi\rangle$ looks like $|\varphi\rangle$?", and the answer drives measurement, fidelity, and every gate to come.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, born_probabilities

np.random.seed(42)
viz.use_course_style()

## 1. Kets, bras, and their overlap

A state $|\psi\rangle$ is a column of amplitudes — a **ket**. Its **bra** $\langle\psi|$ is the *conjugate transpose*: a row, with every amplitude complex-conjugated.

$$ |\psi\rangle = \begin{pmatrix} a_0 \\ a_1 \end{pmatrix}, \qquad \langle\psi| = \begin{pmatrix} a_0^* & a_1^* \end{pmatrix}. $$

The **inner product** $\langle\varphi|\psi\rangle$ multiplies the bra of $\varphi$ into the ket of $\psi$, giving a single complex number. In NumPy this is `phi.conj() @ psi` — equivalently `np.vdot(phi, psi)`, which conjugates its first argument for you.

In [ ]:
phi = KET_PLUS
psi = qubit_state(theta=2 * np.pi / 3, phi=np.pi / 4)

print("bra <+| =", np.round(phi.conj(), 3))
print("<+|psi> =", np.round(np.vdot(phi, psi), 3), " (a single complex number)")
print("by hand =", np.round(phi.conj() @ psi, 3), " (identical)")

**Read the output.** The bra is the row of conjugated amplitudes; contracting it with the ket collapses two vectors into one complex number. `np.vdot` and the explicit `phi.conj() @ psi` agree exactly — conjugation of the *first* factor is the whole point, and it is what makes the next identity work out to a real probability.

## 2. The Born rule is a squared overlap

Picking the amplitude on $|0\rangle$ is itself an inner product: $\langle 0|\psi\rangle = a_0$. So the Born rule from `01/04` is exactly the squared magnitude of an overlap with a basis state:

$$ P(x) = |\langle x|\psi\rangle|^2. $$

Measurement asks "how much does $|\psi\rangle$ overlap each basis state?", and squares the answer.

In [ ]:
p_from_overlap = {x: abs(np.vdot(basis, psi)) ** 2 for x, basis in [("0", KET_0), ("1", KET_1)]}
p_from_born = born_probabilities(psi)
print("P(x) from |<x|psi>|^2 :", {k: round(v, 3) for k, v in p_from_overlap.items()})
print("P(x) from born_probabilities:", {k: round(v, 3) for k, v in p_from_born.items()})

**Read the output.** The two routes to the probabilities agree to the digit. The Born rule was an inner product all along — squaring the overlap of the state with each outcome. This is why the inner product, not the raw amplitude, is the object the course keeps returning to.

## 3. Orthonormality and normalisation

Two relations make the computational basis a *clean* ruler:

$$ \langle 0|0\rangle = \langle 1|1\rangle = 1 \quad(\text{unit length}), \qquad \langle 0|1\rangle = 0 \quad(\text{orthogonal}). $$

A state is **normalised** when $\langle\psi|\psi\rangle = 1$ — the inner-product way of writing the rule $|a_0|^2 + |a_1|^2 = 1$ from `01/01`. And $\{|+\rangle, |-\rangle\}$ is *another* orthonormal basis: the phase that separated them in `01/02` makes them perpendicular.

In [ ]:
print("<0|0> =", np.round(np.vdot(KET_0, KET_0), 3), "   <0|1> =", np.round(np.vdot(KET_0, KET_1), 3))
print("<psi|psi> =", np.round(np.vdot(psi, psi).real, 3), " (normalised)")
print("<+|+> =", np.round(np.vdot(KET_PLUS, KET_PLUS), 3), "   <+|-> =", np.round(np.vdot(KET_PLUS, KET_MINUS), 3))

**Read the output.** Each basis state has unit length and is orthogonal to its partner; $|\psi\rangle$ has $\langle\psi|\psi\rangle = 1$, so it is normalised. The $|+\rangle/|-\rangle$ pair is orthonormal too — confirming that "orthogonal" means *physically distinguishable*, not "points the opposite way on a bar chart".

## 4. Overlap as "how alike"

The magnitude $|\langle\varphi|\psi\rangle|$ runs from $0$ (orthogonal, perfectly distinguishable) to $1$ (identical up to a global phase). It is the natural "how alike are these states?" score. Let's hold $|0\rangle$ fixed and sweep a second state from the north pole down to the equator and on to the south pole (varying the Bloch angle $\theta$), watching the squared overlap.

In [ ]:
thetas = np.linspace(0, np.pi, 200)
overlap_sq = [abs(np.vdot(KET_0, qubit_state(theta=t))) ** 2 for t in thetas]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(thetas, overlap_sq, color=COLORS["quantum"], lw=2.5)
ax.scatter([0, np.pi / 2, np.pi], [1, 0.5, 0], color=COLORS["highlight"], zorder=5, s=60)
for t, lab in [(0, r"$|0\rangle$"), (np.pi / 2, r"$|+\rangle$-latitude"), (np.pi, r"$|1\rangle$")]:
    ax.annotate(lab, (t, abs(np.vdot(KET_0, qubit_state(theta=t))) ** 2),
                textcoords="offset points", xytext=(6, 8), color=COLORS["text"])
ax.set_xlabel(r"Bloch angle $\theta$ of the second state")
ax.set_ylabel(r"$|\langle 0|\psi\rangle|^2$")
ax.set_title("Overlap with |0> as a state rotates away", pad=12)
plt.show()

**Read the figure.** At $\theta=0$ the second state *is* $|0\rangle$ and the squared overlap is $1$. By the equator it has fallen to $0.5$, and at the south pole ($|1\rangle$) it reaches $0$ — orthogonal. This smooth $\cos^2(\theta/2)$ descent is the same number the Born rule reported, and the very quantity `01/14` will turn into the **fidelity** between states. You have built the bridge early.

## Your turn

1. **A third direction.** Build $|+i\rangle = \tfrac{1}{\sqrt2}(|0\rangle + i|1\rangle)$ and compute $|\langle +|+i\rangle|^2$. Is it $0$, $1$, or in between — and what does that say about how distinguishable $|+\rangle$ and $|+i\rangle$ are?
2. **Another orthonormal basis.** Verify $\langle +|+\rangle = 1$ and $\langle +|-\rangle = 0$ directly, then confirm any qubit can be written as $\langle +|\psi\rangle\,|+\rangle + \langle -|\psi\rangle\,|-\rangle$.
3. **Global phase is invisible to the overlap magnitude.** Multiply $|\psi\rangle$ by $e^{i\alpha}$ and show $|\langle\varphi|\psi\rangle|$ is unchanged, even though $\langle\varphi|\psi\rangle$ itself rotates. (This is why `01/02`'s global phase never reaches an experiment.)

## What you built

- You formed bras as conjugate transposes and computed overlaps $\langle\varphi|\psi\rangle$ with `np.vdot`.
- You saw the Born rule *is* a squared overlap, $P(x) = |\langle x|\psi\rangle|^2$, tying `01/04` to one clean operation.
- You used orthonormality, checked normalisation as $\langle\psi|\psi\rangle = 1$, and read $|\langle\varphi|\psi\rangle|$ as a "how alike" score from $0$ to $1$.

This single operation is the grammar of everything ahead — gates, expectation values, and fidelity all speak it. Wonderful groundwork.

## What's next

We can now *act* on states, not only compare them. In `01/06_gates_as_unitary_evolution` we meet the operations that move a qubit — and discover that preserving the inner product you built today is exactly what forces those operations to be *unitary*.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 2.1, Cambridge University Press (2010).
- P. A. M. Dirac, "A new notation for quantum mechanics", *Mathematical Proceedings of the Cambridge Philosophical Society* 35, 416-418 (1939). DOI:10.1017/S0305004100021162

**Previous:** `notebooks/01_QuantumFoundations/04_measurement_and_born_rule.ipynb` · **Next:** `notebooks/01_QuantumFoundations/06_gates_as_unitary_evolution.ipynb`